In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report
from tqdm import tqdm

# CONFIG
MODEL_NAME = "huawei-noah/TinyBERT_General_4L_312D"
MAX_LEN, BATCH_SIZE, EPOCHS, LR = 128, 32, 3, 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

label2id = {'O': 0, 'B-PER':1, 'I-PER':2, 'B-LOC':3, 'I-LOC':4, 'B-ORG':5, 'I-ORG':6, 'B-MISC':7, 'I-MISC':8}
id2label = {v: k for k, v in label2id.items()}

# DATASET
class NERDataset(Dataset):
    def __init__(self, path, tokenizer, max_len):
        self.texts, self.labels = [], []
        with open(path) as f:
            tokens, labels = [], []
            for line in f:
                if line.startswith("-DOCSTART") or line.strip() == "":
                    if tokens:
                        self.texts.append(tokens)
                        self.labels.append(labels)
                        tokens, labels = [], []
                    continue
                splits = line.strip().split()
                tokens.append(splits[0])
                labels.append(label2id[splits[-1]])
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        encodings = self.tokenizer(self.texts[idx], is_split_into_words=True,
                                   return_offsets_mapping=True, padding='max_length',
                                   truncation=True, max_length=self.max_len)
        input_ids = torch.tensor(encodings['input_ids'])
        attention_mask = torch.tensor(encodings['attention_mask'])
        label_ids = torch.full((self.max_len,), -100)
        for i, w in enumerate(encodings.word_ids()):
            if w is not None and w < len(self.labels[idx]):
                label_ids[i] = self.labels[idx][w]
        return input_ids, attention_mask, label_ids

# GENERATOR
class Generator(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state
        logits = self.fc(output)
        return logits

# DISCRIMINATOR
class Discriminator(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.bert.config.hidden_size + num_labels
        self.attention = nn.MultiheadAttention(embed_dim=self.hidden_size, num_heads=3, batch_first=True)
        self.fc = nn.Linear(self.hidden_size, 2)

    def forward(self, input_ids, attention_mask, tag_logits):
        text_embeds = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state
        tag_embeds = F.gumbel_softmax(tag_logits, tau=0.5, hard=False)
        combined = torch.cat([text_embeds, tag_embeds], dim=-1)
        combined = combined[:, :attention_mask.size(1), :]
        attn_output, _ = self.attention(combined, combined, combined)
        logits = self.fc(attn_output)
        return logits

# LOADERS
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_loader = DataLoader(NERDataset("train.txt", tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(NERDataset("valid.txt", tokenizer, MAX_LEN), batch_size=1)
test_loader  = DataLoader(NERDataset("test.txt", tokenizer, MAX_LEN), batch_size=1)

# MODELS + OPTIMIZERS
G = Generator(MODEL_NAME, len(label2id)).to(DEVICE)
D = Discriminator(MODEL_NAME, len(label2id)).to(DEVICE)
optimizer_G = AdamW(G.parameters(), lr=LR)
optimizer_D = AdamW(D.parameters(), lr=LR)

ce_loss = nn.CrossEntropyLoss(ignore_index=-100)

# TRAINING LOOP
for epoch in range(EPOCHS):
    G.train(); D.train()
    for input_ids, attention_mask, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)

        # Generator
        tag_logits = G(input_ids, attention_mask)
        g_loss = ce_loss(tag_logits.view(-1, len(label2id)), labels.view(-1))

        # Discriminator
        d_logits = D(input_ids, attention_mask, tag_logits)
        real_targets = (labels != -100).long()
        d_loss = ce_loss(d_logits.view(-1, 2), real_targets.view(-1))

        # Backward
        optimizer_G.zero_grad(); optimizer_D.zero_grad()
        (g_loss + d_loss).backward()
        optimizer_G.step(); optimizer_D.step()

    print(f"Epoch {epoch+1} | G Loss: {g_loss.item():.4f} | D Loss: {d_loss.item():.4f}")

# EVALUATION FUNCTION
def evaluate_generator(model, loader, name):
    print(f"\n--- {name} Evaluation ---")
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            input_ids, attention_mask = input_ids.to(DEVICE), attention_mask.to(DEVICE)
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            labels = labels.numpy()
            for p_seq, l_seq in zip(preds, labels):
                for p, l in zip(p_seq, l_seq):
                    if l != -100:
                        preds_all.append(id2label[p])
                        labels_all.append(id2label[l])
    print(classification_report(labels_all, preds_all, digits=4))

# FINAL VALIDATION + TEST EVALUATION
evaluate_generator(G, val_loader, "Validation")
evaluate_generator(G, test_loader, "Test")


Epoch 1: 100%|██████████| 439/439 [41:44<00:00,  5.70s/it]


Epoch 1 | G Loss: 0.2257 | D Loss: 0.0003


Epoch 2: 100%|██████████| 439/439 [41:52<00:00,  5.72s/it]


Epoch 2 | G Loss: 0.2643 | D Loss: 0.0001


Epoch 3: 100%|██████████| 439/439 [42:14<00:00,  5.77s/it]


Epoch 3 | G Loss: 0.1060 | D Loss: 0.0000

--- Validation Evaluation ---
              precision    recall  f1-score   support

       B-LOC     0.8711    0.9423    0.9053      2618
      B-MISC     0.7462    0.7522    0.7492      1231
       B-ORG     0.8879    0.7515    0.8140      2056
       B-PER     0.8998    0.9634    0.9305      3029
       I-LOC     0.7246    0.7117    0.7181       281
      I-MISC     0.6236    0.5564    0.5881       390
       I-ORG     0.7772    0.6667    0.7177       900
       I-PER     0.9683    0.9739    0.9711      2757
           O     0.9889    0.9896    0.9893     49658

    accuracy                         0.9647     62920
   macro avg     0.8320    0.8120    0.8204     62920
weighted avg     0.9643    0.9647    0.9641     62920


--- Test Evaluation ---
              precision    recall  f1-score   support

       B-LOC     0.8079    0.9034    0.8530      2123
      B-MISC     0.6713    0.6847    0.6779       996
       B-ORG     0.8291    0.7106 

In [ ]:
class ChunkingDiscriminator(nn.Module):
    def __init__(self, model_name, num_labels, num_heads=8):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.bert.config.hidden_size  # Hidden size of the BERT model
        self.tag_size = num_labels  # Size for tag logits (the number of chunk labels)

        # Calculate the total embedding size
        total_embed_size = self.hidden_size + self.tag_size

        # Ensure the total embedding size is divisible by num_heads
        if total_embed_size % num_heads != 0:
            # Adjust the tag embedding size to match attention layer requirements
            adjustment = num_heads - (total_embed_size % num_heads)
            self.tag_size += adjustment  # Increase the tag size to match

        # Create a multi-head attention layer
        self.attention = nn.MultiheadAttention(embed_dim=self.hidden_size + self.tag_size, num_heads=num_heads, batch_first=True)

        # Fully connected layer for classification output
        self.fc = nn.Linear(self.hidden_size + self.tag_size, 2)  # Binary classification: real or fake

    def forward(self, input_ids, attention_mask, tag_logits):
        # Get BERT embeddings for text
        text_embeds = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state

        # Apply Gumbel-softmax for tag logits to get tag embeddings
        tag_embeds = F.gumbel_softmax(tag_logits, tau=0.5, hard=False)

        # If the tag embedding size was adjusted, pad tag embeddings
        if tag_embeds.size(-1) != self.tag_size:
            pad_size = self.tag_size - tag_embeds.size(-1)
            tag_embeds = F.pad(tag_embeds, (0, pad_size), value=0)

        # Concatenate BERT embeddings and tag embeddings
        combined = torch.cat([text_embeds, tag_embeds], dim=-1)

        # Mask padding in the combined embeddings
        combined = combined[:, :attention_mask.size(1), :]

        # Apply multi-head attention
        attn_output, _ = self.attention(combined, combined, combined)

        # Classification output
        logits = self.fc(attn_output)

        return logits





In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# DYNAMICALLY LOAD CHUNK LABELS FROM BOTH TRAIN AND TEST FILES
def extract_chunk_labels(*paths):
    label_set = set()
    for path in paths:
        with open(path) as f:
            for line in f:
                if line.strip() == "":
                    continue
                splits = line.strip().split()
                label_set.add(splits[2])
    return sorted(label_set)

chunk_labels = extract_chunk_labels("Chunk_train.txt", "Chunk_test.txt")
chunk_label2id = {label: i for i, label in enumerate(chunk_labels)}
chunk_id2label = {i: label for label, i in chunk_label2id.items()}

print("Detected chunk labels:", chunk_labels)

# CHUNKING DATASET CLASS
class ChunkingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        encodings = self.tokenizer(self.texts[idx], is_split_into_words=True,
                                   return_offsets_mapping=True, padding='max_length',
                                   truncation=True, max_length=self.max_len)
        input_ids = torch.tensor(encodings['input_ids'])
        attention_mask = torch.tensor(encodings['attention_mask'])
        label_ids = torch.full((self.max_len,), -100)
        for i, w in enumerate(encodings.word_ids()):
            if w is not None and w < len(self.labels[idx]):
                label_ids[i] = self.labels[idx][w]
        return input_ids, attention_mask, label_ids

# LOAD CHUNK TRAIN DATA AND SPLIT
def load_and_split_chunk_data(path, test_size=0.1):
    texts, labels = [], []
    with open(path) as f:
        tokens, tags = [], []
        for line in f:
            if line.strip() == "":
                if tokens:
                    texts.append(tokens)
                    labels.append([chunk_label2id[t] for t in tags])
                    tokens, tags = [], []
                continue
            splits = line.strip().split()
            tokens.append(splits[0])
            tags.append(splits[2])
    return train_test_split(texts, labels, test_size=test_size, random_state=42)

# LOAD DATA
chunk_train_texts, chunk_val_texts, chunk_train_labels, chunk_val_labels = load_and_split_chunk_data("Chunk_train.txt")

# LOAD CHUNK TEST DATA
chunk_test_texts, chunk_test_labels = [], []
with open("Chunk_test.txt") as f:
    tokens, tags = [], []
    for line in f:
        if line.strip() == "":
            if tokens:
                chunk_test_texts.append(tokens)
                chunk_test_labels.append([chunk_label2id[t] for t in tags])
                tokens, tags = [], []
            continue
        splits = line.strip().split()
        tokens.append(splits[0])
        tags.append(splits[2])

# CHUNKING LOADERS
chunk_train_loader = DataLoader(ChunkingDataset(chunk_train_texts, chunk_train_labels, tokenizer, MAX_LEN),
                                batch_size=BATCH_SIZE, shuffle=True)
chunk_val_loader = DataLoader(ChunkingDataset(chunk_val_texts, chunk_val_labels, tokenizer, MAX_LEN),
                              batch_size=1)
chunk_test_loader = DataLoader(ChunkingDataset(chunk_test_texts, chunk_test_labels, tokenizer, MAX_LEN),
                               batch_size=1)

# GENERATOR & DISCRIMINATOR for CHUNKING
Chunk_G = Generator(MODEL_NAME, len(chunk_label2id)).to(DEVICE)
Chunk_D = ChunkingDiscriminator(MODEL_NAME, len(chunk_label2id)).to(DEVICE)  # Updated discriminator
optimizer_Chunk_G = AdamW(Chunk_G.parameters(), lr=LR)
optimizer_Chunk_D = AdamW(Chunk_D.parameters(), lr=LR)

# TRAINING LOOP for CHUNKING
for epoch in range(EPOCHS):
    Chunk_G.train(); Chunk_D.train()
    for input_ids, attention_mask, labels in tqdm(chunk_train_loader, desc=f"[Chunking] Epoch {epoch+1}"):
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)

        # Generator forward
        tag_logits = Chunk_G(input_ids, attention_mask)
        g_loss = ce_loss(tag_logits.view(-1, len(chunk_label2id)), labels.view(-1))

        # Discriminator forward
        d_logits = Chunk_D(input_ids, attention_mask, tag_logits)
        real_targets = (labels != -100).long()
        d_loss = ce_loss(d_logits.view(-1, 2), real_targets.view(-1))

        # Backward
        optimizer_Chunk_G.zero_grad(); optimizer_Chunk_D.zero_grad()
        (g_loss + d_loss).backward()
        optimizer_Chunk_G.step(); optimizer_Chunk_D.step()

    print(f"[Chunking] Epoch {epoch+1} | G Loss: {g_loss.item():.4f} | D Loss: {d_loss.item():.4f}")

# EVALUATION FUNCTION for CHUNKING
def evaluate_chunking_generator(model, loader, name):
    print(f"\n--- {name} Chunking Evaluation ---")
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            input_ids, attention_mask = input_ids.to(DEVICE), attention_mask.to(DEVICE)
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            labels = labels.numpy()
            for p_seq, l_seq in zip(preds, labels):
                for p, l in zip(p_seq, l_seq):
                    if l != -100:
                        preds_all.append(chunk_id2label[p])
                        labels_all.append(chunk_id2label[l])
    print(classification_report(labels_all, preds_all, digits=4))

# FINAL VALIDATION + TEST EVALUATION
evaluate_chunking_generator(Chunk_G, chunk_val_loader, "Validation")
evaluate_chunking_generator(Chunk_G, chunk_test_loader, "Test")



Detected chunk labels: ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-UCP', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-UCP', 'I-VP', 'O']


[Chunking] Epoch 1: 100%|██████████| 252/252 [24:48<00:00,  5.91s/it]


[Chunking] Epoch 1 | G Loss: 0.4797 | D Loss: 0.0009


[Chunking] Epoch 2:  10%|▉         | 24/252 [02:20<21:45,  5.72s/it]